In [3]:
from google.colab import files
uploaded = files.upload()

Saving final_train.csv to final_train (1).csv
Saving final_validation.csv to final_validation (1).csv
Saving test_input.csv to test_input (1).csv


In [5]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

In [6]:
# ── 1. Load data ─────────────────────────────────────────────────────────────
print("1. Loading data...")
train_df = pd.read_csv("final_train.csv")
val_df   = pd.read_csv("final_validation.csv")
test_df  = pd.read_csv("test_input.csv")

print(f"   Train rows:      {len(train_df)}")
print(f"   Validation rows: {len(val_df)}")
print(f"   Test rows:       {len(test_df)}")

# ── 2. Encode labels ──────────────────────────────────────────────────────────
label_encoder = LabelEncoder()
train_df["encoded_label"] = label_encoder.fit_transform(train_df["label"])
val_df["encoded_label"]   = label_encoder.transform(val_df["label"])
num_labels = len(label_encoder.classes_)

# ── 3. Build HuggingFace Datasets ─────────────────────────────────────────────
train_dataset = Dataset.from_pandas(
    train_df[["sentence", "encoded_label"]].rename(columns={"encoded_label": "label"})
)
val_dataset = Dataset.from_pandas(
    val_df[["sentence", "encoded_label"]].rename(columns={"encoded_label": "label"})
)
test_dataset = Dataset.from_pandas(test_df[["sentence"]])

# ── 4. Tokenize ───────────────────────────────────────────────────────────────
print("2. Loading tokenizer and model (bert-base-uncased)...")
model_name = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

print("3. Tokenizing data...")
def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val   = val_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function, batched=True)

# ── 5. Metrics ────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = float((predictions == labels).mean())
    return {"accuracy": accuracy}

# ── 6. Training arguments ─────────────────────────────────────────────────────
print("4. Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# ── 7. Train ──────────────────────────────────────────────────────────────────
print("5. Fine-tuning BERT (this will take a few minutes on a GPU)...")
trainer.train()

# ── 8. Predict on test set ────────────────────────────────────────────────────
print("6. Making predictions on the test set...")
predictions_output = trainer.predict(tokenized_test)
predicted_classes  = np.argmax(predictions_output.predictions, axis=1)
final_labels       = label_encoder.inverse_transform(predicted_classes)

# ── 9. Save predictions ───────────────────────────────────────────────────────
print("7. Saving predictions.csv...")
submission_df = test_df.copy()
submission_df["label"] = final_labels
submission_df = submission_df[["sentence", "label"]]
submission_df.to_csv("predictions.csv", index=False)

print("Done! predictions.csv is ready.")

1. Loading data...
   Train rows:      12104
   Validation rows: 4035
   Test rows:       4035
2. Loading tokenizer and model (bert-base-uncased)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

6. Making predictions on the test set...


7. Saving predictions.csv...
Done! predictions.csv is ready.
